In [1]:
import os


In [2]:
%pwd

'/mnt/d/dl_project/research'

In [3]:
os.chdir("../")
%pwd

'/mnt/d/dl_project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

In [5]:
@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_learning_rate: float

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories
import tensorflow as tf

I0000 00:00:1782231013.521923   26760 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782231013.649309   26760 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782231039.405468   26760 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath =CONFIG_FILE_PATH,
                 params_filepath =PARAMS_FILE_PATH,
                 schema_filepath =SCHEMA_FILE_PATH
                 ):
        self.config =read_yaml(config_filepath)
        self.params =read_yaml(params_filepath)
        self.schema =read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params =self.params
        create_directories([training.root_dir])

        training_config = TrainingConfig(
            root_dir =Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data =Path(training.training_data),
            params_epochs =params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation = params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate = params.LEARNING_RATE
        )

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
tf.config.run_functions_eagerly(True)

In [ ]:
# component
class Training:
    def __init__(self, config=ConfigurationManager):
        self.config = config

    def get_base_model(self):
        self.model =tf.keras.models.load_model(self.config.updated_base_model_path)

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation='bilinear',
            class_mode='categorical'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory= self.config.training_data,
            subset = "validation",
            shuffle=False,
            **dataflow_kwargs
                            )
        
        if self.config.params_is_augmentation:
            
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range = 40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )

        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory =self.config.training_data,
            subset='training',
            shuffle=True,
            **dataflow_kwargs
        )
        print(self.train_generator.class_indices)

    @staticmethod
    def save_model(path:Path, model: tf.keras.Model):
            model.save(path)

    
    def train(self, callbacks=list):
        tf.keras.backend.clear_session()
        self.get_base_model()
        optimizer = tf.keras.optimizers.SGD(learning_rate=self.config.params_learning_rate)
        self.model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy', # Matches your class_mode='categorical'
            metrics=["accuracy"]
        )
        
        checkpoint = tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.trained_model_path, # Path where the best model will be saved
            monitor='val_loss',
            save_best_only=True,                    # Only save if validation loss improves
            verbose=1
        )
        
        early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=8, # Stop after 3 epochs of no improvement
        restore_best_weights=True
         )
        
        lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.2, 
            patience=3,
            verbose=1
        )
        
        
        my_callbacks = [checkpoint,lr_scheduler, early_stopping]

        self.model.fit(
             self.train_generator,
             epochs=self.config.params_epochs,
             validation_data=self.valid_generator,
             callbacks=my_callbacks
             )
        
        
        self.save_model(
             path=self.config.trained_model_path,
             model=self.model)
         



In [ ]:
try:
    config = ConfigurationManager()
    training_config = config.get_model_training_config()
    training = Training(config = training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e


[2026-06-23 16:10:58,558: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-23 16:10:58,565: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-23 16:10:58,571: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-23 16:10:58,575: INFO: common: created directory at: artifacts]
[2026-06-23 16:10:58,582: INFO: common: created directory at: artifacts/training]


I0000 00:00:1782231059.551087   26760 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3539 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


[2026-06-23 16:11:02,708: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.
{'Cyst': 0, 'Normal': 1, 'Stone': 2, 'Tumor': 3}
[2026-06-23 16:11:05,275: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Epoch 1/30


/mnt/d/dl_project/venv/lib/python3.10/site-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(
I0000 00:00:1782231065.944088   26760 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1782231066.303226   26760 cuda_dnn.cc:461] Loaded cuDNN version 92302


623/623 ━━━━━━━━━━━━━━━━━━━━ 276s 434ms/step - accuracy: 0.5298 - loss: 8.4965 - val_accuracy: 0.5854 - val_loss: 8.2339 - learning_rate: 0.0010
Epoch 2/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 284s 455ms/step - accuracy: 0.6746 - loss: 7.9113 - val_accuracy: 0.5983 - val_loss: 8.0805 - learning_rate: 0.0010
Epoch 3/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 285s 457ms/step - accuracy: 0.7146 - loss: 7.6279 - val_accuracy: 0.6253 - val_loss: 7.8972 - learning_rate: 0.0010
Epoch 4/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 282s 452ms/step - accuracy: 0.7494 - loss: 7.3779 - val_accuracy: 0.6220 - val_loss: 7.7546 - learning_rate: 0.0010
Epoch 5/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 288s 462ms/step - accuracy: 0.7719 - loss: 7.1427 - val_accuracy: 0.6084 - val_loss: 7.6251 - learning_rate: 0.0010
Epoch 6/30
623/623 ━━━━━━━━━━━━━━━━━━━━ 285s 458ms/step - accuracy: 0.7833 - loss: 6.9564 - val_accuracy: 0.6212 - val_loss: 7.4349 - learning_rate: 0.0010
Epoch 7/30
619/623 ━━━━━━━━━━━━━━━━━━━━ 1s 390ms/step - accuracy: 0.7967 - 